In [1]:
from langchain.tools import tool
from langchain_groq import ChatGroq
import requests, pathlib
from langchain_community.utilities import SQLDatabase
from pydantic import Field, create_model
from langchain_core.output_parsers import PydanticOutputParser

from enum import Enum
from typing import Optional
import json
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.messages import AIMessage, HumanMessage

model = ChatGroq(model="qwen/qwen3-32b", temperature=0)

/Users/jessicacardoso/Projects/langchain-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
local_path = pathlib.Path("Chinook.db")

if local_path.exists():
    print(f"{local_path} already exists, skipping download.")
else:
    response = requests.get(url)
    if response.status_code == 200:
        local_path.write_bytes(response.content)
        print(f"File downloaded and saved as {local_path}")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")

Chinook.db already exists, skipping download.


In [3]:
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

print(f"Dialect: {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")
print(f'Sample output: {db.run("SELECT * FROM Artist LIMIT 5;")}')

Dialect: sqlite
Available tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
Sample output: [(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]


In [4]:
@tool
def sql_db_list_tables() -> str:
    """List all available tables in the SQL database."""
    return ", ".join(db.get_usable_table_names())


@tool
def sql_db_schema(tables: str) -> str:
    """Get the schema information for the specified tables.
    The input is a comma-separated list of table names.
    """
    names = [name.strip() for name in tables.split(",")]
    return db.get_table_info(names)


@tool
def sql_db_query(query: str) -> str:
    """Execute the provided SQL query on the database.
    Always return the result of the query.
    """
    return str(db.run(query))


my_tools = [sql_db_list_tables, sql_db_schema, sql_db_query]

In [5]:
def create_agent_models_and_prompt(tools: list, dialect: str):
    # Create tool descriptions for the prompt
    tools_descriptions = []
    for t in tools:
        # Get the schema of arguments and ESCAPE curly braces for LangChain prompt
        args_json = json.dumps(t.args, indent=2).replace("{", "{{").replace("}", "}}")
        desc = f"- Name: '{t.name}'\n  Purpose: {t.description}\n  Arguments Schema: {args_json}"
        tools_descriptions.append(desc)

    tools_desc_str = "\n".join(tools_descriptions)

    # Dynamic Pydantic Models
    tool_names = {t.name: t.name for t in tools}
    DynamicToolEnum = Enum("ToolName", tool_names)

    ActionModel = create_model(
        "Action",
        action_name=(
            DynamicToolEnum,
            Field(..., description="The name of the tool to execute."),
        ),
        parameters=(
            dict,
            Field(default_factory=dict, description="Dictionary of arguments."),
        ),
    )

    NextTurnModel = create_model(
        "NextTurn",
        thought=(str, Field(..., description="Reasoning for this step.")),
        actions=(
            list[ActionModel],
            Field(default_factory=list, description="List of tool actions."),
        ),
        finish=(bool, Field(False, description="True if answer is ready.")),
        final_answer=(
            Optional[str],
            Field(None, description="The final response to user."),
        ),
    )

    parser = PydanticOutputParser(pydantic_object=NextTurnModel)
    format_instructions = (
        parser.get_format_instructions().replace("{", "{{").replace("}", "}}")
    )

    system_prompt = f"""You are a SQL Architect using dialect: {dialect}.

AVAILABLE TOOLS:
{tools_desc_str}

CRITICAL RULES:
1. You MUST respond ONLY with a valid JSON object.
2. Use the following schema for your response:
{format_instructions}

3. Do not include markdown code blocks (like ```json) in your output.
4. If 'finish' is True, 'final_answer' must be populated.
"""

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            MessagesPlaceholder(variable_name="agent_messages"),
            ("user", "{question}"),
        ]
    )

    return prompt, NextTurnModel

In [6]:
def execute_agent_loop(question: str, tools: list, max_turns: int = 10):
    print(f"--- Starting Agent: {question} ---\n")

    tools_map = {tool.name: tool for tool in tools}
    prompt, NextTurnStructure = create_agent_models_and_prompt(tools, db.dialect)

    # Use json_mode for structured output adherence
    runnable_agent = prompt | model.with_structured_output(
        NextTurnStructure, method="json_mode"
    )

    agent_history = []

    for turn_id in range(1, max_turns + 1):
        print(f"--- Turn {turn_id} ---")

        try:
            response = runnable_agent.invoke(
                {"question": question, "agent_messages": agent_history}
            )
        except Exception as e:
            print(f"Error: {e}")
            break

        print(f"Thought: {response.thought}")

        if response.finish:
            print(f"\n✅ Final Answer: {response.final_answer}")
            return response.final_answer

        observations = []
        for action in response.actions:
            t_name = (
                action.action_name.value
                if hasattr(action.action_name, "value")
                else action.action_name
            )
            t_params = action.parameters
            print(f"Action: {t_name}({t_params})")

            tool_fn = tools_map.get(t_name)
            if tool_fn:
                try:
                    result = tool_fn.invoke(t_params)
                    obs = f"Observation from {t_name}: {result}"
                except Exception as e:
                    obs = f"Error in {t_name}: {str(e)}"
            else:
                obs = f"Error: Tool {t_name} not found."

            observations.append(obs)
            print(f"Observation: {obs[:100]}...")

        # Update history with serialized response and observations
        agent_history.append(AIMessage(content=response.model_dump_json()))
        for o in observations:
            agent_history.append(HumanMessage(content=o))

In [7]:
q = "Which genre on average has the longest tracks?"
execute_agent_loop(q, my_tools, 20)

--- Starting Agent: Which genre on average has the longest tracks? ---

--- Turn 1 ---
Thought: To determine which genre has the longest average tracks, I need to analyze the database tables. First, I'll list the available tables to understand the structure. Then, I'll check their schemas to identify columns related to genres and track durations. Finally, I'll execute a query to calculate the average track length per genre.
Action: sql_db_list_tables({})
Observation: Observation from sql_db_list_tables: Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine,...
--- Turn 2 ---
Thought: Now that I know the tables exist, I need to check the schema of the 'Genre' and 'Track' tables to identify the relevant columns (e.g., GenreId, TrackId, Milliseconds). This will help me write a query to calculate average track duration per genre.
Action: sql_db_schema({'tables': 'Genre,Track'})
Observation: Observation from sql_db_schema: 
CREATE TABLE "Genre" (
	"GenreId" INTEGER NOT NULL, 
	"Nam

"The genre with the longest average tracks is 'Sci Fi & Fantasy' with an average duration of approximately 48.5 minutes."